# PHI Screening and HIPAA Compliance Pipeline
### Clinical Notes De-identification Notebook

**Purpose:** Takes a CSV of clinical notes and runs a three-layer PHI screening pipeline to produce a clean, de-identified output CSV ready for ML training.

**Input:** `notes_raw.csv` with columns: `visit_id`, `note_text`

**Output:** `notes_clean.csv` with columns: `visit_id`, `note_text_clean`, `phi_entities_found`, `residual_phi_flag`

**Three layers:**
1. Microsoft Presidio -- standard PHI (names, dates, locations, phone, SSN, email)
2. Custom clinical regex -- MRN, pager numbers, room/bed numbers, clinical date formats
3. Residual audit flag -- flags notes needing manual review after cleaning

**HIPAA note:** De-identification is required even for in-house ML models. Using PHI to train a model is secondary use of health information subject to HIPAA. No in-house exception exists. Safe Harbor de-identification removes all 18 PHI identifier categories.

## 1. Setup and Installation

In [ ]:
# Install if needed -- run once, then restart kernel
# !pip install presidio-analyzer presidio-anonymizer --quiet
# !python -m spacy download en_core_web_lg --quiet

import pandas as pd
import numpy as np
import re
import json
import warnings
warnings.filterwarnings('ignore')

from presidio_analyzer import AnalyzerEngine
from presidio_anonymizer import AnonymizerEngine
from presidio_anonymizer.entities import OperatorConfig

print('Libraries loaded.')
import presidio_analyzer
print(f'presidio-analyzer version: {presidio_analyzer.__version__}')

## 2. Load Input Data

Change `INPUT_PATH` to point to your notes file.

In [ ]:
INPUT_PATH  = 'notes_raw.csv'
OUTPUT_PATH = 'notes_clean.csv'

notes_df = pd.read_csv(INPUT_PATH)
print(f'Loaded: {notes_df.shape[0]:,} notes')
print(f'Columns: {notes_df.columns.tolist()}')

# Validate required columns exist
for col in ['visit_id', 'note_text']:
    if col not in notes_df.columns:
        raise ValueError(f'Required column "{col}" not found in input file.')

null_count = notes_df['note_text'].isna().sum()
print(f'Notes with null text: {null_count}')
if null_count > 0:
    print('  Warning: null notes will be skipped.')

print()
print('Sample note (first 300 chars):')
print(notes_df['note_text'].iloc[0][:300])

## 3. Layer 1 -- Microsoft Presidio

Presidio detects and replaces standard PHI categories:
- **PERSON** -- patient and physician names
- **DATE_TIME** -- admission dates, birth dates, procedure dates
- **LOCATION** -- hospital names, addresses, cities
- **PHONE_NUMBER** -- phone and fax numbers
- **EMAIL_ADDRESS, US_SSN, URL, IP_ADDRESS** -- additional identifiers
- **MEDICAL_LICENSE, US_DRIVER_LICENSE** -- professional and personal IDs

Each detected entity is replaced with a readable `[PLACEHOLDER]` tag.

In [ ]:
analyzer   = AnalyzerEngine()
anonymizer = AnonymizerEngine()

PRESIDIO_ENTITIES = [
    'PERSON', 'DATE_TIME', 'LOCATION', 'PHONE_NUMBER',
    'EMAIL_ADDRESS', 'US_SSN', 'URL', 'IP_ADDRESS',
    'US_DRIVER_LICENSE', 'MEDICAL_LICENSE', 'CREDIT_CARD',
]

def run_presidio(text):
    """
    Run Presidio analysis and anonymization on a single note.
    Returns: (cleaned_text, list_of_entity_types_found)
    """
    if pd.isna(text) or str(text).strip() == '':
        return text, []

    results = analyzer.analyze(
        text     = str(text),
        language = 'en',
        entities = PRESIDIO_ENTITIES
    )

    entities_found = list(set([r.entity_type for r in results]))

    if not results:
        return text, []

    anonymized = anonymizer.anonymize(
        text             = str(text),
        analyzer_results = results,
        operators = {
            'PERSON':            OperatorConfig('replace', {'new_value': '[NAME]'}),
            'DATE_TIME':         OperatorConfig('replace', {'new_value': '[DATE]'}),
            'LOCATION':          OperatorConfig('replace', {'new_value': '[LOCATION]'}),
            'PHONE_NUMBER':      OperatorConfig('replace', {'new_value': '[PHONE]'}),
            'EMAIL_ADDRESS':     OperatorConfig('replace', {'new_value': '[EMAIL]'}),
            'US_SSN':            OperatorConfig('replace', {'new_value': '[SSN]'}),
            'URL':               OperatorConfig('replace', {'new_value': '[URL]'}),
            'IP_ADDRESS':        OperatorConfig('replace', {'new_value': '[IP]'}),
            'US_DRIVER_LICENSE': OperatorConfig('replace', {'new_value': '[ID]'}),
            'MEDICAL_LICENSE':   OperatorConfig('replace', {'new_value': '[LICENSE]'}),
            'CREDIT_CARD':       OperatorConfig('replace', {'new_value': '[CARD]'}),
        }
    )

    return anonymized.text, entities_found


# Test on first note
sample    = notes_df['note_text'].iloc[0]
l1_text, l1_entities = run_presidio(sample)

print('Layer 1 Presidio -- test result:')
print(f'Entities detected: {l1_entities}')
print()
print('ORIGINAL (first 300 chars):')
print(sample[:300])
print()
print('AFTER PRESIDIO (first 300 chars):')
print(l1_text[:300])

## 4. Layer 2 -- Custom Clinical Regex

Catches clinical-specific identifiers that Presidio commonly misses:
- Medical Record Numbers (MRN) in various formats
- Pager and provider contact numbers
- Room and bed numbers (patient location identifiers)
- Date formats in clinical shorthand (01/15/23, 15-Jan-23)
- NPI numbers (National Provider Identifier)

Layer 2 runs on the already-cleaned output from Layer 1.

In [ ]:
# Custom patterns -- each entry: (regex_pattern, replacement, description)
# Note: patterns use raw strings (r'...') to handle backslashes correctly
CLINICAL_PATTERNS = [

    # Medical record numbers in common formats
    (r'\bMRN[:\s#]*\d{4,10}\b',               '[MRN]',    'Medical record number'),
    (r'\bMedical\s+Record[:\s#]*\d{4,10}\b',  '[MRN]',    'Medical record number spelled out'),
    (r'\bChart[:\s#]*\d{4,10}\b',              '[MRN]',    'Chart number'),

    # Pager and provider contact numbers
    (r'\b[Pp]ager[:\s#]*\d{4,7}\b',            '[PAGER]',  'Pager number'),
    (r'\b[Pp]g[:\s#]*\d{4,7}\b',               '[PAGER]',  'Pager number abbreviated'),

    # Room and bed numbers
    (r'\b[Rr]oom\s*\d{1,4}[A-Za-z]?\b',        '[ROOM]',   'Room number'),
    (r'\b[Bb]ed\s*\d{1,4}[A-Za-z]?\b',         '[BED]',    'Bed number'),
    (r'\b[Rr]m\.?\s*\d{1,4}[A-Za-z]?\b',      '[ROOM]',   'Room abbreviation'),

    # Date formats in clinical shorthand
    (r'\b\d{1,2}/\d{1,2}/\d{2,4}\b',          '[DATE]',   'Date MM/DD/YY'),
    (r'\b\d{1,2}-\d{1,2}-\d{2,4}\b',          '[DATE]',   'Date MM-DD-YY'),

    # NPI and account numbers
    (r'\b[Nn][Pp][Ii][:\s#]*\d{10}\b',          '[NPI]',    'National Provider Identifier'),
    (r'\b[Aa]ccount[:\s#]*\d{5,12}\b',          '[ACCOUNT]','Account number'),
]


def run_custom_regex(text):
    """
    Apply custom clinical regex patterns on already-Presidio-cleaned text.
    Returns: (cleaned_text, list_of_additional_entities_found)
    """
    if pd.isna(text) or str(text).strip() == '':
        return text, []

    text  = str(text)
    extra = []

    for pattern, replacement, description in CLINICAL_PATTERNS:
        if re.search(pattern, text, flags=re.IGNORECASE):
            extra.append(description)
            text = re.sub(pattern, replacement, text, flags=re.IGNORECASE)

    return text, extra


# Test on Layer 1 output
l2_text, l2_extra = run_custom_regex(l1_text)
print('Layer 2 Custom Regex -- test result:')
print(f'Additional PHI caught: {l2_extra if l2_extra else "None -- Presidio already covered"}')
print()
print('AFTER LAYER 2 (first 300 chars):')
print(l2_text[:300])

## 5. Layer 3 -- Residual PHI Audit Flag

Scans cleaned text for patterns that may still contain PHI after Layers 1 and 2.

**Important:** This layer does NOT remove anything. It flags notes for human review. A flagged note is not automatically excluded -- it means a person should verify before including it in ML training.

Notes are flagged if they contain:
- Title + capitalized name patterns (Dr. Smith, Mr. Johnson)
- Two consecutive capitalized words that could be a name
- Digit sequences resembling phone numbers or IDs
- Long digit sequences that could be MRN or account numbers

In [ ]:
RESIDUAL_CHECKS = [

    # Title + capitalized name -- 'Dr. Smith', 'Mr. Johnson', 'RN Williams'
    (r'\b(Dr|Mr|Mrs|Ms|Prof|RN|MD|DO|NP|PA)\.?\s+[A-Z][a-z]+\b',
     'Possible name with title'),

    # Two consecutive capitalized words not inside a placeholder bracket
    (r'(?<!\[)\b[A-Z][a-z]{2,}\s+[A-Z][a-z]{2,}\b',
     'Possible name -- two capitalized words'),

    # Phone number pattern not yet replaced
    (r'(?<!\[)\b\d{3}[-.]\d{3}[-.]\d{4}\b',
     'Possible phone number'),

    # Long digit sequence -- possible MRN or account number
    (r'(?<!\[)\b\d{7,12}\b',
     'Long digit sequence -- possible ID'),

    # Full written-out date not caught by earlier layers
    (r'\b(January|February|March|April|May|June|July|August'
     r'|September|October|November|December)\s+\d{1,2},?\s+\d{4}\b',
     'Possible full written date'),
]


def check_residual_phi(text):
    """
    Scan cleaned text for remaining PHI patterns.
    Returns: (needs_review: bool, reasons: list)
    """
    if pd.isna(text) or str(text).strip() == '':
        return False, []

    text  = str(text)
    flags = []

    for pattern, description in RESIDUAL_CHECKS:
        if re.search(pattern, text):
            flags.append(description)

    return len(flags) > 0, flags


# Test on Layer 2 output
needs_review, reasons = check_residual_phi(l2_text)
print('Layer 3 Residual Audit -- test result:')
print(f'Needs manual review: {needs_review}')
if reasons:
    print(f'Reasons: {reasons}')
else:
    print('No residual PHI patterns detected -- note is clean.')

## 6. Run Full Pipeline on All Notes

Applies all three layers to every note in the dataset. Progress is printed every 500 notes.

In [ ]:
def process_note(note_text):
    """
    Run all three PHI screening layers on a single note.
    Returns a dictionary with clean text, entities found, and audit flag.
    """
    # Layer 1 -- Presidio
    text_l1, entities_l1 = run_presidio(note_text)

    # Layer 2 -- Custom clinical regex
    text_l2, entities_l2 = run_custom_regex(text_l1)

    # Layer 3 -- Residual audit
    needs_review, residual = check_residual_phi(text_l2)

    # Combine all detected entities into one list
    all_entities = entities_l1 + entities_l2
    if residual:
        all_entities += [f'RESIDUAL: {r}' for r in residual]

    return {
        'note_text_clean':    text_l2,
        'phi_entities_found': json.dumps(all_entities) if all_entities else '[]',
        'residual_phi_flag':  1 if needs_review else 0
    }


print(f'Processing {len(notes_df):,} notes through PHI pipeline...')
print('This may take several minutes for large datasets -- Presidio NER is computationally intensive.')
print()

results = []
for i, row in notes_df.iterrows():
    result              = process_note(row['note_text'])
    result['visit_id']  = row['visit_id']
    results.append(result)

    if (i + 1) % 500 == 0:
        print(f'  {i+1:,} / {len(notes_df):,} processed...')

print(f'Done. {len(results):,} notes processed.')

## 7. Review Results

In [ ]:
output_df = pd.DataFrame(results)[
    ['visit_id', 'note_text_clean', 'phi_entities_found', 'residual_phi_flag']
]

print('=== PHI Pipeline Summary ===')
print(f'Total notes processed:           {len(output_df):,}')

notes_with_phi = output_df[output_df['phi_entities_found'] != '[]']
print(f'Notes with PHI detected:         {len(notes_with_phi):,}  '
      f'({len(notes_with_phi)/len(output_df)*100:.1f}%)')

flagged = output_df['residual_phi_flag'].sum()
print(f'Flagged for manual review:       {flagged:,}  '
      f'({flagged/len(output_df)*100:.1f}%)')

# Entity type breakdown
all_entities = []
for e in output_df['phi_entities_found']:
    all_entities.extend(json.loads(e))

if all_entities:
    print()
    print('PHI entity types found:')
    counts = pd.Series(all_entities).value_counts()
    for entity, count in counts.head(15).items():
        print(f'  {entity:<45} {count:>6}')

In [ ]:
# Show a before/after comparison for a note where PHI was detected
if len(notes_with_phi) > 0:
    sample_row  = notes_with_phi.iloc[0]
    original    = notes_df[notes_df['visit_id'] == sample_row['visit_id']]['note_text'].values[0]

    print('=== Before / After Comparison ===')
    print()
    print('ORIGINAL (first 400 chars):')
    print(original[:400])
    print()
    print('CLEANED (first 400 chars):')
    print(sample_row['note_text_clean'][:400])
    print()
    print(f'PHI removed:     {sample_row["phi_entities_found"]}')
    print(f'Residual flag:   {sample_row["residual_phi_flag"]}')

In [ ]:
# Show flagged notes that need manual review
flagged_df = output_df[output_df['residual_phi_flag'] == 1]

if len(flagged_df) > 0:
    print(f'=== {len(flagged_df)} Notes Flagged for Manual Review ===')
    print('Review these before including in ML training.')
    print()
    for _, row in flagged_df.head(5).iterrows():
        print(f'visit_id: {row["visit_id"]}')
        print(f'PHI found: {row["phi_entities_found"]}')
        print(f'Cleaned text (first 250 chars): {row["note_text_clean"][:250]}')
        print()
else:
    print('No notes flagged -- all notes passed residual audit.')

## 8. Save Output

In [ ]:
output_df.to_csv(OUTPUT_PATH, index=False)

print(f'Saved: {OUTPUT_PATH}')
print(f'  Rows:    {len(output_df):,}')
print(f'  Columns: {output_df.columns.tolist()}')
print()
print('Column descriptions:')
print('  visit_id            -- join key to feature matrix (same visit_id as all other data)')
print('  note_text_clean     -- de-identified note text ready for ClinicalBERT fine-tuning')
print('  phi_entities_found  -- JSON list of all PHI types detected and removed')
print('  residual_phi_flag   -- 1 = manual review recommended before training')
print()
clean_count = (output_df['residual_phi_flag'] == 0).sum()
print(f'Notes ready for ML training (no residual flag): {clean_count:,}')
print(f'Notes requiring manual review:                  {flagged:,}')
print()
print('Next step: use note_text_clean column as input to ClinicalBERT fine-tuning pipeline.')